# ETL Pipeline - Azure Data Lake Medallion Architecture

##### A comprehensive ETL pipeline leveraging Azure Data Lake Storage Gen2 and Databricks for data transformation across Bronze, Silver, and Gold layers using the Medallion Architecture pattern. 

## Phase 1: Establishing Connection Between Databricks and Azure


### 1. Create Workspace Client and Secret Scope

Initialize the Databricks Workspace Client and create a secret scope to securely store Azure Data Lake access keys.

In [0]:
# creation of the secret scope
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

wc = WorkspaceClient()
try:
    wc.secrets.create_scope(scope="olist_scope")
    print("Secret scope 'olist_scope' successfully created!")
except ResourceAlreadyExists:
    print("Secret scope 'olist_scope' already exists - skipping creation.")

Secret scope 'olist_scope' already exists - skipping creation.


### 2. Clone GitHub Repository and Create Environment File

Clone the project repository into Databricks workspace and create a `.env` file to store sensitive configuration values (storage account names, container paths, and access keys).

### 3. Load Configuration from Environment File

Load environment variables from the `.env` file using `load_dotenv()` to retrieve storage account names, container names, and authentication keys.

In [0]:
# resources configuration
from dotenv import load_dotenv
import os
load_dotenv()

STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
CONTAINER = os.getenv("CONTAINER")
KEY_VAULT = os.getenv("KEY_VAULT")

SECRET_KEY = dbutils.secrets.get("olist_scope", "adls_key")
KEY_URL = f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net"
print("all the resources loaded successfully!")


all the resources loaded successfully!


### 4. Store Azure Access Key in Databricks Secret Scope

Securely store the Azure Data Lake access key in the Databricks secret scope, loaded from the environment configuration.

In [0]:
# saving the key in the vault 
wc.secrets.put_secret(
    scope="olist_scope",
    key="adls_key",
    string_value= KEY_VAULT
)
print("Key securely saved in the vault!")

Key securely saved in the vault!


### 5. Create Azure Path Generation Function

Define a reusable function to generate authenticated ABFSS paths for accessing Azure Data Lake Storage folders.

In [0]:
def get_azure_path(path=""):
    """Generate an authenticated abfss:// path for Azure Data Lake"""
    base_path = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
    if path:
        return f"{base_path}/{path}"
    return base_path

print("Azure Storage access configured!")

Azure Storage access configured!


## Phase 2: Loading Data from Bronze Layer

Load raw data files from the Bronze layer in Azure Data Lake Storage container into Spark DataFrames for transformation.

### 1. Define Table Names and Path Dictionary

Create a list of table names and build a dictionary mapping each table to its corresponding Azure Storage path in the Bronze layer.

In [0]:
tables = ['customer',
         'sellers',
         'geolocation', 
         'products',
         'orders',
         'order_items',
         'order_payments',
         'order_reviews']

full_azure_path = {}

for table in tables:
    bronze_path = f"bronze/olist_{table}.parquet"
    full_azure_path[table] = get_azure_path(bronze_path)
print(f"full azure paths saved seccussufully in :\n {full_azure_path}")


full azure paths saved seccussufully in :
 {'customer': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_customer.parquet', 'sellers': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_sellers.parquet', 'geolocation': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_geolocation.parquet', 'products': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_products.parquet', 'orders': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_orders.parquet', 'order_items': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_order_items.parquet', 'order_payments': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_order_payments.parquet', 'order_reviews': 'abfss://olist-warehouse-container@datawarehouseolist.dfs.core.windows.net/bronze/olist_

### 2. Load Datasets Dynamically from Parquet Files

Dynamically load all datasets from Azure Storage as Parquet files into a dictionary of Spark DataFrames.

In [0]:
datasets = {}

for dataset_name, azure_path_url in full_azure_path.items():
    datasets[dataset_name] = (
                                spark.read
                                     .format("parquet")
                                     .option(KEY_URL, SECRET_KEY)
                                     .load(azure_path_url)
                                )
    print(f'"{dataset_name}" dataset loaded successfully!')

print('All datasets loaded successfully')

"customer" dataset loaded successfully!
"sellers" dataset loaded successfully!
"geolocation" dataset loaded successfully!
"products" dataset loaded successfully!
"orders" dataset loaded successfully!
"order_items" dataset loaded successfully!
"order_payments" dataset loaded successfully!
"order_reviews" dataset loaded successfully!
All datasets loaded successfully


### 3. Assign DataFrames to Named Variables

Convert the loaded datasets dictionary into individual DataFrame variables with consistent naming convention (`df_<table_name>`).

In [0]:
for dataset_name, table in datasets.items():
    globals()[f'df_{dataset_name}'] = table
    print(f'"{dataset_name}" dataframe created successfully!')

print('all dataframes created successfully')
df_customer.show(4)

"customer" dataframe created successfully!
"sellers" dataframe created successfully!
"geolocation" dataframe created successfully!
"products" dataframe created successfully!
"orders" dataframe created successfully!
"order_items" dataframe created successfully!
"order_payments" dataframe created successfully!
"order_reviews" dataframe created successfully!
all dataframes created successfully
+--------------------+------------------------+-------------+--------------+
|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+------------------------+-------------+--------------+
|f7d7fc0a59ef4363f...|                   69900|   rio branco|            AC|
|5dbba6c01268a8ad4...|                   69900|   rio branco|            AC|
|36b1c0516f123351f...|                   69900|   rio branco|            AC|
|721d1092e1a6460c6...|                   69900|   rio branco|            AC|
+--------------------+------------------------+-------------+------

## Phase 3: Transforming Data for Silver Layer

Apply data quality transformations, type casting, and schema optimization to prepare cleaned datasets for the Silver layer.

### 1. Customer Table Transformation

Process and transform the customer dimension table with proper data types, cleaning, and schema optimization.

#### 1.1. Inspect Original Schema

Examine the current schema and data types of the raw customer table from the Bronze layer.

In [0]:
df_customer.printSchema()

root
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



#### 1.2. Apply Data Transformations

Clean, cast data types, trim whitespace, and restructure the customer table with optimized column names and formats.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col 
from pyspark.sql.types import StringType, IntegerType

df_customer = df_customer.select(
    F.trim(F.col("customer_unique_id")).cast(StringType())\
                                       .alias("customer_id"),
    F.col("customer_zip_code_prefix").cast(IntegerType())\
                                    .alias("zip_code_prefix"), 
    F.trim(F.col("customer_city")).cast(StringType())\
                                  .alias("customer_city"),
    F.upper(F.trim(F.col("customer_state"))).cast(StringType())\
                                            .alias("customer_state")
)

datasets['customer'] = df_customer

df_customer.show(5)

+--------------------+---------------+-------------+--------------+
|         customer_id|zip_code_prefix|customer_city|customer_state|
+--------------------+---------------+-------------+--------------+
|f7d7fc0a59ef4363f...|          69900|   rio branco|            AC|
|5dbba6c01268a8ad4...|          69900|   rio branco|            AC|
|36b1c0516f123351f...|          69900|   rio branco|            AC|
|721d1092e1a6460c6...|          69900|   rio branco|            AC|
|7a2dc4682890550eb...|          69900|   rio branco|            AC|
+--------------------+---------------+-------------+--------------+
only showing top 5 rows


#### 1.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_customer.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



### 2. Seller Table Transformation

Process and transform the seller dimension table with proper data types, cleaning, and schema optimization.

#### 2.1. Inspect Original Schema

Examine the current schema and data types of the raw seller table from the Bronze layer.

In [0]:
df_sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



#### 2.2. Apply Data Transformations

Clean, cast data types, trim whitespace, and restructure the seller table with optimized column names and formats.

In [0]:
df_sellers = df_sellers.select(
    F.trim(F.col("seller_id")).cast(StringType())\
                              .alias("seller_id"),
    F.col("seller_zip_code_prefix").cast(IntegerType())\
                                   .alias("zip_code_prefix"), 
    F.trim(F.col("seller_city")).cast(StringType())\
                                 .alias("seller_city"),
    F.upper(F.trim(F.col("seller_state"))).cast(StringType())\
                                          .alias("seller_state")
)
datasets['sellers'] = df_sellers
df_sellers.show(5)

+--------------------+---------------+-----------+------------+
|           seller_id|zip_code_prefix|seller_city|seller_state|
+--------------------+---------------+-----------+------------+
|c13ef0cfbe42f1907...|           1207|  sao paulo|          SP|
|5444b12c82f21c923...|           2051|  sao paulo|          SP|
|1cbd32d00d01bb808...|           3363|  sao paulo|          SP|
|71593c7413973a1e1...|           3407|  sao paulo|          SP|
|6f1a1263039c76e68...|           3581|  sao paulo|          SP|
+--------------------+---------------+-----------+------------+
only showing top 5 rows


#### 2.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



### 3. Geolocation Table Transformation

Process and transform the geolocation dimension table with proper data types, cleaning, and schema optimization.

#### 3.1. Inspect Original Schema

Examine the current schema and data types of the raw geolocation table from the Bronze layer.

In [0]:
df_geolocation.printSchema()

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- geolocation_state_name: string (nullable = true)
 |-- geolocation_region: string (nullable = true)



#### 3.2. Apply Data Transformations

Clean, cast data types, trim whitespace, and restructure the geolocation table with optimized column names and formats.

In [0]:
from pyspark.sql.types import DoubleType
df_geolocation = df_geolocation.select(
    F.col("geolocation_zip_code_prefix").cast(IntegerType())\
                                        .alias("zip_code_prefix"),
    F.col("geolocation_lat").cast(DoubleType())\
                            .alias("latitude"),
    F.col("geolocation_lng").cast(DoubleType())\
                            .alias("longitude"),
    F.trim(F.col("geolocation_city")).cast(StringType())\
                                     .alias("city"),
    F.upper(F.trim(F.col("geolocation_state"))).cast(StringType())\
                                             .alias("state_postal"),
    F.upper(F.trim(F.col("geolocation_state_name"))).cast(StringType())\
                                                   .alias("state"),
    F.initcap(F.trim(F.col("geolocation_region"))).cast(StringType())\
                                                .alias("region")
                                             )

datasets['geolocation'] = df_geolocation

df_geolocation.show(5)


+---------------+-------------------+------------------+---------+------------+---------+---------+
|zip_code_prefix|           latitude|         longitude|     city|state_postal|    state|   region|
+---------------+-------------------+------------------+---------+------------+---------+---------+
|           1037| -23.54533514931352|-46.63899521271491|sao paulo|          SP|SÃO PAULO|Southeast|
|           1046| -23.54588497157669|-46.64374420942636|sao paulo|          SP|SÃO PAULO|Southeast|
|           1041|-23.544036423155262|-46.63988866258875|sao paulo|          SP|SÃO PAULO|Southeast|
|           1035|-23.541612840173386|-46.64158346016764|sao paulo|          SP|SÃO PAULO|Southeast|
|           1012| -23.54780758661154|-46.63484393782474|são paulo|          SP|SÃO PAULO|Southeast|
+---------------+-------------------+------------------+---------+------------+---------+---------+
only showing top 5 rows


#### 3.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_geolocation.printSchema()

root
 |-- zip_code_prefix: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city: string (nullable = true)
 |-- state_postal: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)



### 4. Products Table Transformation

Process and transform the products dimension table with proper data types, cleaning, and schema optimization.

#### 4.1. Inspect Original Schema

Examine the current schema and data types of the raw products table from the Bronze layer.

In [0]:
df_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_length: string (nullable = true)
 |-- product_description_length: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



#### 4.2. Apply Data Transformations

Clean, cast data types, trim whitespace, and restructure the products table with optimized column names and formats.

In [0]:
df_products.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_length|product_description_length|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|a0ab96e461d745377...|         climatizacao|                 41|                       717|                 1|          1050.0|             18.0|              7.0|             8.0|
|20ae7c024ede613f4...|       telefonia_fixa|                 25|                       455|                 1|           330.0|             17.0|             11.0|             9.0|
|4d7585daba2f8b3ed...|       telefonia_fixa|                 53|                       897|    

In [0]:
from pyspark.sql.functions import col, trim, initcap, regexp_replace
from pyspark.sql.types import StringType, IntegerType, FloatType

df_products = df_products.select(
    trim(col("product_id")).cast(StringType()).alias("product_id"),
    initcap(trim(regexp_replace(col("product_category_name"), '_', ' ')))\
                 .cast(StringType())\
                 .alias("category_name"),
    col("product_name_length").cast(IntegerType()).alias("name_length"),
    col("product_description_length").cast(IntegerType())\
                                     .alias("description_length"),
    col("product_photos_qty").cast(IntegerType()).alias("photos_quantity"),
    
    col("product_weight_g").cast(FloatType()).alias("weight_grams"),
    col("product_length_cm").cast(FloatType()).alias("length_cm"),
    col("product_height_cm").cast(FloatType()).alias("height_cm"),
    col("product_width_cm").cast(FloatType()).alias("width_cm")
)

# Store the clean DataFrame back into the dictionary
datasets['products'] = df_products

df_products.show(5)

+--------------------+--------------------+-----------+------------------+---------------+------------+---------+---------+--------+
|          product_id|       category_name|name_length|description_length|photos_quantity|weight_grams|length_cm|height_cm|width_cm|
+--------------------+--------------------+-----------+------------------+---------------+------------+---------+---------+--------+
|a0ab96e461d745377...|        Climatizacao|         41|               717|              1|      1050.0|     18.0|      7.0|     8.0|
|20ae7c024ede613f4...|      Telefonia Fixa|         25|               455|              1|       330.0|     17.0|     11.0|     9.0|
|4d7585daba2f8b3ed...|      Telefonia Fixa|         53|               897|              2|       300.0|     15.0|      8.0|     9.0|
|ad7aebed205805125...|Construcao Ferram...|         41|              2526|              2|      1150.0|     22.0|     10.0|     9.0|
|980ecbcc15fe174ec...|Agro Industria E ...|         48|              

#### 4.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- name_length: integer (nullable = true)
 |-- description_length: integer (nullable = true)
 |-- photos_quantity: integer (nullable = true)
 |-- weight_grams: float (nullable = true)
 |-- length_cm: float (nullable = true)
 |-- height_cm: float (nullable = true)
 |-- width_cm: float (nullable = true)



### 5. Orders Table Transformation

Process and transform the orders fact table with proper data types, timestamp handling, and schema optimization.

#### 5.1. Inspect Original Schema

Examine the current schema and data types of the raw orders table from the Bronze layer.

In [0]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_date: string (nullable = true)
 |-- order_approved_date: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



#### 5.2. Apply Data Transformations

Clean, cast data types, convert timestamps, and restructure the orders table with optimized column names and formats.

In [0]:
from pyspark.sql.types import TimestampType

df_orders = df_orders.select(
    trim(col("order_id")).cast(StringType())\
                         .alias("order_id"),
    trim(col("customer_unique_id")).cast(StringType())\
                            .alias("customer_id"),
    initcap(trim(col("order_status"))).cast(StringType())\
                                      .alias("order_status"),
    col("order_purchase_date").cast(TimestampType())\
                                .alias("purchase_date"),
    col("order_approved_date").cast(TimestampType())\
                                .alias("approved_date"),
    col("order_delivered_carrier_date").cast(TimestampType())\
                                         .alias("carrier_date"),
    col("order_delivered_customer_date").cast(TimestampType())\
                                          .alias("delivered_date"),
    col("order_estimated_delivery_date").cast(TimestampType())\
                                          .alias("estimated_delivery_date")
)

datasets['orders'] = df_orders

df_orders.show(5)

+--------------------+--------------------+------------+-------------------+-------------------+------------+--------------+-----------------------+
|            order_id|         customer_id|order_status|      purchase_date|      approved_date|carrier_date|delivered_date|estimated_delivery_date|
+--------------------+--------------------+------------+-------------------+-------------------+------------+--------------+-----------------------+
|a2e4c44360b4a57bd...|7b9d52d22310baeca...|    Approved|2017-02-06 20:18:17|2017-02-06 20:30:19|        NULL|          NULL|    2017-03-01 00:00:00|
|132f1e724165a07f6...|6a068ccd3a149b5c8...|    Approved|2017-04-25 01:25:34|2017-04-30 20:32:41|        NULL|          NULL|    2017-05-22 00:00:00|
|809a282bbd5dbcabb...|009b0127b727ab0ba...|    Canceled|2016-09-13 15:24:19|2016-10-07 13:16:46|        NULL|          NULL|    2016-09-30 00:00:00|
|e5215415bb6f76fe3...|281096eb031de8c31...|    Canceled|2016-10-22 08:25:27|               NULL|        NU

#### 5.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- approved_date: timestamp (nullable = true)
 |-- carrier_date: timestamp (nullable = true)
 |-- delivered_date: timestamp (nullable = true)
 |-- estimated_delivery_date: timestamp (nullable = true)



### 6. Order Items Table Transformation

Process and transform the order items fact table with proper data types, numeric precision, and schema optimization.

#### 6.1. Inspect Original Schema

Examine the current schema and data types of the raw order items table from the Bronze layer.

In [0]:
df_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



#### 6.2. Apply Data Transformations

Clean, cast data types, handle numeric precision, and restructure the order items table with optimized column names and formats.

In [0]:
from pyspark.sql.types import DecimalType

df_order_items = df_order_items.select(
    trim(col("order_id")).cast(StringType())\
                         .alias("order_id"),
    col("order_item_id").cast(IntegerType())\
                        .alias("item_id"),
    trim(col("product_id")).cast(StringType())\
                           .alias("product_id"),
    trim(col("seller_id")).cast(StringType())\
                          .alias("seller_id"),
    col("shipping_limit_date").cast(TimestampType())\
                              .alias("shipping_limit_date"),
    col("price").cast(DecimalType(10, 2))\
                .alias("price"),
    col("freight_value").cast(DecimalType(10, 2))\
                        .alias("freight_value")
)

datasets['order_items'] = df_order_items

df_order_items.show(5)

+--------------------+-------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------+--------------------+--------------------+-------------------+-----+-------------+
|3ee6513ae7ea23bdf...|      1|8a3254bee785a526d...|96804ea39d96eb908...|2018-05-04 03:55:26| 0.85|        18.23|
|6e864b3f0ec710311...|      1|8a3254bee785a526d...|96804ea39d96eb908...|2018-05-02 20:30:34| 0.85|        18.23|
|c5bdd8ef3c0ec4202...|      2|8a3254bee785a526d...|96804ea39d96eb908...|2018-05-07 02:55:22| 0.85|        22.30|
|8272b63d03f5f79c5...|      2|05b515fdc76e888aa...|2709af9587499e95e...|2017-07-21 18:25:23| 1.20|         7.89|
|8272b63d03f5f79c5...|      3|05b515fdc76e888aa...|2709af9587499e95e...|2017-07-21 18:25:23| 1.20|         7.89|
+--------------------+-------+--------------------+--------------------+-------------------+----

#### 6.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)



### 7. Order Payments Table Transformation

Process and transform the order payments fact table with proper data types, numeric precision, and schema optimization.

#### 7.1. Inspect Original Schema

Examine the current schema and data types of the raw order payments table from the Bronze layer.

In [0]:
df_order_payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



#### 7.2. Apply Data Transformations

Clean, cast data types, handle numeric precision, and restructure the order payments table with optimized column names and formats.

In [0]:
df_order_payments = df_order_payments.select(
    trim(col("order_id")).cast(StringType())\
                         .alias("order_id"),
    col("payment_sequential").cast(IntegerType())\
                             .alias("payment_sequential"),
    initcap(trim(col("payment_type"))).cast(StringType())\
                                      .alias("payment_type"),
    col("payment_installments").cast(IntegerType())\
                               .alias("payment_installments"),
    col("payment_value").cast(DecimalType(10, 2))\
                        .alias("payment_value")
)

datasets['order_payments'] = df_order_payments

df_order_payments.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|744bade1fcf9ff3f3...|                 2| Credit_card|                   0|        58.69|
|1a57108394169c0b4...|                 2| Credit_card|                   0|       129.94|
|8bcbe01d44d147f90...|                 4|     Voucher|                   1|         0.00|
|fa65dad1b0e818e3c...|                14|     Voucher|                   1|         0.00|
|6ccb433e00daae128...|                 4|     Voucher|                   1|         0.00|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


#### 7.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_order_payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(10,2) (nullable = true)



### 8. Order Reviews Table Transformation

Process and transform the order reviews fact table with proper data types, text handling, and schema optimization.

#### 8.1. Inspect Original Schema

Examine the current schema and data types of the raw order reviews table from the Bronze layer.

In [0]:
df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_date: string (nullable = true)



#### 8.2. Apply Data Transformations

Clean, cast data types, handle timestamps and text fields, and restructure the order reviews table with optimized column names and formats.

In [0]:
df_order_reviews = df_order_reviews.select(
    trim(col("review_id")).cast(StringType())\
                          .alias("review_id"),
    trim(col("order_id")).cast(StringType())\
                         .alias("order_id"),
    col("review_score").cast(IntegerType())\
                       .alias("review_score"),
    col("review_creation_date").cast(TimestampType())\
                                 .alias("creation_date"),
    col("review_answer_date").cast(TimestampType())\
                             .alias("answer_date")
)

datasets['order_reviews'] = df_order_reviews

df_order_reviews.show(5)

+--------------------+--------------------+------------+-------------------+-------------------+
|           review_id|            order_id|review_score|      creation_date|        answer_date|
+--------------------+--------------------+------------+-------------------+-------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|2018-01-18 00:00:00|2018-01-18 21:46:00|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|2018-03-10 00:00:00|2018-03-11 03:05:00|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|2018-02-17 00:00:00|2018-02-18 14:36:00|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|2017-04-21 00:00:00|2017-04-21 22:02:00|
|f7c4243c7fe1938f1...|8e6bfb81e283fa7e4...|           5|2018-03-01 00:00:00|2018-03-02 10:26:00|
+--------------------+--------------------+------------+-------------------+-------------------+
only showing top 5 rows


#### 8.3. Verify Transformed Schema

Validate the final schema after transformations to ensure all data types and structures are correctly applied.

In [0]:
df_order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- creation_date: timestamp (nullable = true)
 |-- answer_date: timestamp (nullable = true)



### 9. Export Transformed Tables to Silver Layer

Write all transformed DataFrames to the Silver layer in Azure Data Lake Storage as Delta format files for optimized query performance.

In [0]:
for i, (df_name, df) in enumerate(datasets.items()):
    silver_path = get_azure_path(f"/silver/{df_name}.delta")
    df.write.mode('overwrite')\
            .format('delta')\
            .option(KEY_URL, SECRET_KEY)\
            .save(silver_path)
    print(f"{df_name.capitalize()}.delta file saved seccussfully!\n")        
print(f'All {i+1} files have been saved successfully!')

Customer.delta file saved seccussfully!

Sellers.delta file saved seccussfully!

Geolocation.delta file saved seccussfully!

Products.delta file saved seccussfully!

Orders.delta file saved seccussfully!

Order_items.delta file saved seccussfully!

Order_payments.delta file saved seccussfully!

Order_reviews.delta file saved seccussfully!

All 8 files have been saved successfully!


## Phase 4: Prepare Data for Gold Layer

Create business-ready aggregated datasets and analytical views optimized for reporting and business intelligence.

### 1. Load Transformed Data from Silver Layer

Read all Delta format tables from the Silver layer back into Spark DataFrames for Gold layer processing.

In [0]:
# Dictionary to store loaded DataFrames
silver_datasets = {}

# Load each Delta table from Silver layer
for table_name in tables:
    silver_path = get_azure_path(f"silver/{table_name}.delta")
    silver_datasets[table_name] = (
        spark.read
             .format("delta")
             .option(KEY_URL, SECRET_KEY)
             .load(silver_path)
    )
    print(f'"{table_name}" Delta table loaded successfully!')

print('\nAll Silver layer tables loaded successfully!')

"customer" Delta table loaded successfully!
"sellers" Delta table loaded successfully!
"geolocation" Delta table loaded successfully!
"products" Delta table loaded successfully!
"orders" Delta table loaded successfully!
"order_items" Delta table loaded successfully!
"order_payments" Delta table loaded successfully!
"order_reviews" Delta table loaded successfully!

All Silver layer tables loaded successfully!


### 2. Assign to global DataFrame variables

In [0]:

for i, (table_name, df) in enumerate(silver_datasets.items()) :
    globals()[f'df_{table_name}'] = df
    print(f'df_{table_name} assigned successfully!')

print(f'\nAll {i+1} DataFrames ready for Gold layer transformations!')

df_customer assigned successfully!
df_sellers assigned successfully!
df_geolocation assigned successfully!
df_products assigned successfully!
df_orders assigned successfully!
df_order_items assigned successfully!
df_order_payments assigned successfully!
df_order_reviews assigned successfully!

All 8 DataFrames ready for Gold layer transformations!


### 3. Create Dimension Tables

Transform Silver layer tables into star schema dimension tables with business keys, derived metrics, and optimized structures for analytical queries.

#### 3.1. Create Customer Dimension Table

Build dim_customers with sequential business keys (CTM_0000001 format) and clean customer attributes.

In [0]:

dims = {}
from pyspark.sql import Window
# 1. Define window specification
window_spec = Window.orderBy("customer_id")

# 2. Assign the sequential CTM_ format directly in one shot
dim_customers = df_customer\
    .withColumn(
                "customer_seq_id",
                 F.concat(
                    F.lit("CTM_"),
                    F.format_string("%07d", F.row_number().over(window_spec))
                ))

# 3. Clean, transform, cast, and structure dimension table
dim_customers = dim_customers.select(
    trim(col("customer_seq_id")).cast(StringType())\
                                .alias("customer_seq_id"),
    col("customer_id").alias("customer_identifier_id"),
    "zip_code_prefix", 
    "customer_city",
    "customer_state"
    )
dims['customers'] = dim_customers

dim_customers.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------------+----------------------+---------------+-------------+--------------+
|customer_seq_id|customer_identifier_id|zip_code_prefix|customer_city|customer_state|
+---------------+----------------------+---------------+-------------+--------------+
|    CTM_0000001|  0000366f3b9a7992b...|           7787|      cajamar|            SP|
|    CTM_0000002|  0000b849f77a49e4a...|           6053|       osasco|            SP|
|    CTM_0000003|  0000f46a3911fa3c0...|          88115|     sao jose|            SC|
|    CTM_0000004|  0000f6ccb0745a6a4...|          66812|        belem|            PA|
|    CTM_0000005|  0004aac84e0df4da2...|          18040|     sorocaba|            SP|
+---------------+----------------------+---------------+-------------+--------------+
only showing top 5 rows


#### 3.2. Create Seller Dimension Table

Build dim_sellers with sequential business keys (SLR_0000001 format) and clean seller location attributes.


In [0]:
window_spec = Window.orderBy("seller_id")

dim_sellers = df_sellers\
    .withColumn(
                "seller_seq_id",
                 F.concat(
                    F.lit("SLR_"),
                    F.format_string("%07d", F.row_number().over(window_spec))
                ))

dim_sellers = dim_sellers.select(
    trim(col("seller_seq_id")).cast(StringType())\
                                .alias("seller_seq_id"),
    col("seller_id").alias("seller_identifier_id"),
    "zip_code_prefix", 
    "seller_city",
    "seller_state"
    )

dims['sellers'] = dim_sellers

dim_sellers.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+--------------------+---------------+-----------+------------+
|seller_seq_id|seller_identifier_id|zip_code_prefix|seller_city|seller_state|
+-------------+--------------------+---------------+-----------+------------+
|  SLR_0000001|0015a82c2db000af6...|           9080|santo andre|          SP|
|  SLR_0000002|001cca7ae9ae17fb1...|          29156|  cariacica|          ES|
|  SLR_0000003|001e6ad469a905060...|          24754|sao goncalo|          RJ|
|  SLR_0000004|002100f778ceb8431...|          14405|     franca|          SP|
|  SLR_0000005|003554e2dce176b55...|          74565|    goiania|          GO|
+-------------+--------------------+---------------+-----------+------------+
only showing top 5 rows


#### 3.3. Create Geolocation Dimension Table

Build dim_geolocation with geographic coordinates and regional information.


In [0]:
dim_geolocation = df_geolocation.select(
    "zip_code_prefix",
    "latitude",
    "longitude",
    "city",
    "state",
    "state_postal",
    "region")
    
dims['geolocation'] = dim_geolocation

dim_geolocation.show(5)

+---------------+-------------------+------------------+---------+---------+------------+---------+
|zip_code_prefix|           latitude|         longitude|     city|    state|state_postal|   region|
+---------------+-------------------+------------------+---------+---------+------------+---------+
|           1037| -23.54533514931352|-46.63899521271491|sao paulo|SÃO PAULO|          SP|Southeast|
|           1046| -23.54588497157669|-46.64374420942636|sao paulo|SÃO PAULO|          SP|Southeast|
|           1041|-23.544036423155262|-46.63988866258875|sao paulo|SÃO PAULO|          SP|Southeast|
|           1035|-23.541612840173386|-46.64158346016764|sao paulo|SÃO PAULO|          SP|Southeast|
|           1012| -23.54780758661154|-46.63484393782474|são paulo|SÃO PAULO|          SP|Southeast|
+---------------+-------------------+------------------+---------+---------+------------+---------+
only showing top 5 rows


#### 3.4. Create Products Dimension Table

Build dim_products with sequential business keys (PRD_0000001 format), product attributes, and calculated volume/density metrics.


In [0]:
window_spec = Window.orderBy("product_id")

dim_products = df_products\
    .withColumn(
        "product_seq_id",
        F.concat(
            F.lit("PRD_"),
            F.format_string("%07d", F.row_number().over(window_spec))
        ))\
    .withColumn("product_volum_L", 
        F.round(
            (col("height_cm") * col("width_cm") * col("length_cm")) / 1000, 2
        ))\
    .withColumn("product_density_kg_L", 
        F.when(col("product_volum_L") != 0,
            F.round(col("weight_grams") / (1000 * col("product_volum_L")), 2)
        ).otherwise(None))

dim_products = dim_products.select(
    F.trim(col("product_seq_id")).cast(StringType()).alias("product_seq_id"),
    col("product_id").alias('product_identifier_id'),
    "category_name",
    "name_length",
    "description_length",
    "photos_quantity",
    "weight_grams",
    "length_cm",
    "height_cm",
    "width_cm",
    "product_volum_L",          
    "product_density_kg_L"      
)
dims['products'] = dim_products

dim_products.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+---------------------+--------------------+-----------+------------------+---------------+------------+---------+---------+--------+---------------+--------------------+
|product_seq_id|product_identifier_id|       category_name|name_length|description_length|photos_quantity|weight_grams|length_cm|height_cm|width_cm|product_volum_L|product_density_kg_L|
+--------------+---------------------+--------------------+-----------+------------------+---------------+------------+---------+---------+--------+---------------+--------------------+
|   PRD_0000001| 00066f42aeeb9f300...|          Perfumaria|         53|               596|              6|       300.0|     20.0|     16.0|    16.0|           5.12|                0.06|
|   PRD_0000002| 00088930e925c41fd...|          Automotivo|         56|               752|              4|      1225.0|     55.0|     10.0|    26.0|           14.3|                0.09|
|   PRD_0000003| 0009406fd7479715e...|     Cama Mesa Banho|         50

#### 3.5. Create Orders Dimension Table

Build dim_orders with sequential business keys (ORD_0000001 format) and calculated time metrics for order lifecycle stages (approval time, shipment time, carrier time, delivery period, and delivery promise deviation).

In [0]:
window_spec = Window.orderBy("order_id")

dim_orders = df_orders\
    .withColumn("order_seq_id",
        F.concat(
            F.lit("ORD_"),
            F.format_string("%07d", F.row_number().over(window_spec))
        ))\
    .withColumn("approved_time_hours",
        F.round(
            (F.unix_timestamp(col("approved_date")) - F.unix_timestamp(col("purchase_date"))) / 3600, 2
        ))\
    .withColumn("shipment_time_hours",
        F.round(
            (F.unix_timestamp(col("carrier_date")) - F.unix_timestamp(col("approved_date"))) / 3600, 2
        ))\
    .withColumn("carrier_time_days",
        F.round(
            (F.unix_timestamp(col("delivered_date")) - F.unix_timestamp(col("carrier_date"))) / (3600*24), 2   
        ))\
    .withColumn("delivery_period_days",  
        F.round(
            (F.unix_timestamp(col("delivered_date")) - F.unix_timestamp(col("purchase_date"))) / (3600*24), 2   
        ))\
    .withColumn(
        "delivery_promise_deviation_days",
        F.round(
            (F.unix_timestamp(col("estimated_delivery_date")) - F.unix_timestamp(col("delivered_date"))) / (3600*24), 2
    ))\
    .select(
        F.trim(col("order_seq_id")).cast(StringType()).alias("order_seq_id"),
        col("order_id").alias("order_identifier_id"),
        'customer_id',
        "order_status",
        "purchase_date",
        "approved_date",
        "carrier_date",
        "delivered_date",
        "estimated_delivery_date",
        "approved_time_hours",           
        "shipment_time_hours",           
        "carrier_time_days",             
        "delivery_period_days",          
        "delivery_promise_deviation_days" 
    )

dims['orders'] = dim_orders

dim_orders.show(5)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+--------------------+--------------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------------+-------------------+-------------------+-----------------+--------------------+-------------------------------+
|order_seq_id| order_identifier_id|         customer_id|order_status|      purchase_date|      approved_date|       carrier_date|     delivered_date|estimated_delivery_date|approved_time_hours|shipment_time_hours|carrier_time_days|delivery_period_days|delivery_promise_deviation_days|
+------------+--------------------+--------------------+------------+-------------------+-------------------+-------------------+-------------------+-----------------------+-------------------+-------------------+-----------------+--------------------+-------------------------------+
| ORD_0000001|00010242fe8c5a6d1...|871766c5855e863f6...|   Delivered|2017-09-13 08:59:02|2017-09-13 09:45:35|2017-09-19 18:34:16|2017-09-20 23:43

### 4. Create Facts Tables :

#### 4.1. Create Order Items Fact Table

Build fact_order_items with order line items and calculated total charged amounts (price + freight).


In [0]:
facts = {}
fact_order_items = df_order_items\
    .withColumn("total_charged",
        F.round((col("price") + col("freight_value")), 2)
    )\
    .select(
        "order_id",
        "item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value",
        "total_charged"
    )

facts['order_items'] = fact_order_items

fact_order_items.show(5)

+--------------------+-------+--------------------+--------------------+-------------------+-----+-------------+-------------+
|            order_id|item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|total_charged|
+--------------------+-------+--------------------+--------------------+-------------------+-----+-------------+-------------+
|3ee6513ae7ea23bdf...|      1|8a3254bee785a526d...|96804ea39d96eb908...|2018-05-04 03:55:26| 0.85|        18.23|        19.08|
|6e864b3f0ec710311...|      1|8a3254bee785a526d...|96804ea39d96eb908...|2018-05-02 20:30:34| 0.85|        18.23|        19.08|
|c5bdd8ef3c0ec4202...|      2|8a3254bee785a526d...|96804ea39d96eb908...|2018-05-07 02:55:22| 0.85|        22.30|        23.15|
|8272b63d03f5f79c5...|      2|05b515fdc76e888aa...|2709af9587499e95e...|2017-07-21 18:25:23| 1.20|         7.89|         9.09|
|8272b63d03f5f79c5...|      3|05b515fdc76e888aa...|2709af9587499e95e...|2017-07-21 18:25:23| 1.20|         7.89

#### 4.2. Create Order Payments Fact Table

Build fact_order_payments with payment details including payment type, installments, and payment values.

In [0]:
fact_order_payments = df_order_payments\
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    )

facts['order_payments'] = fact_order_payments

fact_order_payments.show(5)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|744bade1fcf9ff3f3...|                 2| Credit_card|                   0|        58.69|
|1a57108394169c0b4...|                 2| Credit_card|                   0|       129.94|
|8bcbe01d44d147f90...|                 4|     Voucher|                   1|         0.00|
|fa65dad1b0e818e3c...|                14|     Voucher|                   1|         0.00|
|6ccb433e00daae128...|                 4|     Voucher|                   1|         0.00|
+--------------------+------------------+------------+--------------------+-------------+
only showing top 5 rows


#### 4.3. Create Order Reviews Fact Table

Build fact_order_reviews with review details and calculated response time metrics.

In [0]:
fact_order_reviews = df_order_reviews\
    .withColumn("response_time_days",
        F.round(
            (F.unix_timestamp(col('answer_date')) - F.unix_timestamp(col("creation_date")))/(3600*20), 2
            ))\
    .select(
        "review_id",
        "order_id",
        "review_score",
        "creation_date",
        "answer_date",
        "response_time_days"
    )

facts['order_reviews'] = fact_order_reviews

fact_order_reviews.show(5)

+--------------------+--------------------+------------+-------------------+-------------------+------------------+
|           review_id|            order_id|review_score|      creation_date|        answer_date|response_time_days|
+--------------------+--------------------+------------+-------------------+-------------------+------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|2018-01-18 00:00:00|2018-01-18 21:46:00|              1.09|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|2018-03-10 00:00:00|2018-03-11 03:05:00|              1.35|
|228ce5500dc1d8e02...|f9e4b658b201a9f2e...|           5|2018-02-17 00:00:00|2018-02-18 14:36:00|              1.93|
|e64fb393e7b32834b...|658677c97b385a9be...|           5|2017-04-21 00:00:00|2017-04-21 22:02:00|               1.1|
|f7c4243c7fe1938f1...|8e6bfb81e283fa7e4...|           5|2018-03-01 00:00:00|2018-03-02 10:26:00|              1.72|
+--------------------+--------------------+------------+----------------

### 5. Save Dimension and Fact Tables to Gold Layer

Write all dimension and fact tables to the Gold layer in Delta format for optimized analytics and reporting.

In [0]:
for dim_or_fact, dims_or_facts in {'dim' : dims, 'fact' : facts}.items():
    for i, (table_name, table_df) in enumerate(dims_or_facts.items()):
        gold_path = get_azure_path(f'/gold/{dim_or_fact}_{table_name}.delta')
        table_df.write.mode('overwrite')\
                        .format('delta')\
                        .option(KEY_URL, SECRET_KEY)\
                        .save(gold_path)
        print(f"{dim_or_fact.capitalize()}_{table_name}.delta file saved successfully to gold!\n")        
    print(f'All {i+1} {dim_or_fact} tables have been saved successfully to gold layer!\n')
    
print('All dims and facts tables have been saved successfully to gold layer!')

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dim_customers.delta file saved successfully to gold!

Dim_sellers.delta file saved successfully to gold!

Dim_geolocation.delta file saved successfully to gold!

Dim_products.delta file saved successfully to gold!

Dim_orders.delta file saved successfully to gold!

All 5 dim tables have been saved successfully to gold layer!

Fact_order_items.delta file saved successfully to gold!

Fact_order_payments.delta file saved successfully to gold!

Fact_order_reviews.delta file saved successfully to gold!

All 3 fact tables have been saved successfully to gold layer!

All dims and facts tables have been saved successfully to gold layer!


### 6. Export Gold Tables to Local Workspace as Parquet

In [0]:
# Define local workspace path for gold data export
load_dotenv()
export_path = os.getenv('PATH_TO_GOLD_DATA_FOLDER')

# Create directory if it doesn't exist
import os
import pyarrow as pa
import pyarrow.parquet as pq
os.makedirs(export_path, exist_ok=True)

# Export all dimension tables
print("\nEXPORTING DIMENSION TABLES...\n")
for table_name, table_df in dims.items():
    output_path = f"{export_path}/dim_{table_name}.parquet"
    
    # Convert to pandas then to pyarrow table and write
    pandas_df = table_df.toPandas()
    arrow_table = pa.Table.from_pandas(pandas_df, preserve_index=False)
    pq.write_table(arrow_table, output_path)
    
    # Get row count for verification
    row_count = len(pandas_df)
    print(f"dim_{table_name}.parquet exported successfully! ({row_count:,} rows)")

print(f"\nAll {len(dims)} dimension tables exported!")

# Export all fact tables
print("\nEXPORTING FACT TABLES...\n")
for table_name, table_df in facts.items():
    output_path = f"{export_path}/fact_{table_name}.parquet"
    
    # Convert to pandas then to pyarrow table and write
    pandas_df = table_df.toPandas()
    arrow_table = pa.Table.from_pandas(pandas_df, preserve_index=False)
    pq.write_table(arrow_table, output_path)

    # Get row count for verification
    row_count = len(pandas_df)
    print(f"fact_{table_name}.parquet exported successfully! ({row_count:,} rows)")

print(f"\nAll {len(facts)} fact tables exported!")

print("\n" + "=" * 80)
print(f"\nSUCCESS! All {len(dims) + len(facts)} gold layer tables exported to workspace folder!")
print(f"Location: {export_path}")


EXPORTING DIMENSION TABLES...



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_customers.parquet exported successfully! (96,096 rows)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_sellers.parquet exported successfully! (3,095 rows)
dim_geolocation.parquet exported successfully! (19,177 rows)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_products.parquet exported successfully! (32,951 rows)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_orders.parquet exported successfully! (99,441 rows)

All 5 dimension tables exported!

EXPORTING FACT TABLES...

fact_order_items.parquet exported successfully! (112,650 rows)
fact_order_payments.parquet exported successfully! (103,886 rows)
fact_order_reviews.parquet exported successfully! (99,224 rows)

All 3 fact tables exported!


SUCCESS! All 8 gold layer tables exported to workspace folder!
Location: /Workspace/Users/rachidkherbech@gmail.com/simplon-data-analyst-training-program-projects/Week 20 - Olist Brazilian E-commerce dataset Full Analysis/data/gold_data


### Done